# TAMPO Algorithms — Colab Test Run Notebook

This notebook tests the currently implemented TAMPO meta-RL algorithms (TAMPO-GCN and TAMPO-LSTM).
Run cells **top to bottom** in order.

## 0. Setup — Clone repo & install dependencies

In [3]:
# ── 0a. Clone the repo ────────────────────────────────────────────
!git clone -b gcn-pyg https://github.com/vikkesh/tampo.git tampo
%cd tampo

Cloning into 'tampo'...
remote: Enumerating objects: 3699, done.
remote: Counting objects: 100% (158/158), done.
remote: Compressing objects: 100% (100/100), done.
remote: Total 3699 (delta 85), reused 114 (delta 46), pack-reused 3541 (from 2)
Receiving objects: 100% (3699/3699), 200.31 MiB | 31.35 MiB/s, done.
Resolving deltas: 100% (114/114), done.
/content/tampo/tampo/tampo


In [ ]:
#if you want to pull latest commits

# 1. Fetch the latest changes from GitHub
!git fetch origin

# 2. Checkout your branch (just in case you aren't on it)
!git checkout gcn-pyg

# 3. Hard reset to exactly match the latest code on that branch
!git reset --hard origin/gcn-pyg


In [4]:
!nvidia-smi

Fri Jun 12 11:56:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
# ── 0b. Install all dependencies ─────────────────────────────────
!pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 83.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.8/269.8 kB 29.0 MB/s eta 0:00:00


In [ ]:
# ── 0c. Verify imports ────────────────────────────────────────────
import torch, gymnasium as gym, yaml, json, numpy as np
from torch_geometric.data import Batch, Data
print("torch:", torch.__version__)
print("gymnasium:", gym.__version__)
print("CUDA available:", torch.cuda.is_available())


## 1. Unit Test — DAGEncoder with Bi-GCN path

Verifies that the encoder:
- Accepts a PyG Batch of **variable-size** graphs
- Produces a context vector of the correct shape 
- Properly handles the bi-directional pass without crashing

In [6]:
import sys, os
sys.path.insert(0, '.')  # tampo root

import torch
from torch_geometric.data import Batch, Data
from algorithms.rl.tampo import DAGEncoder

TASK_FEATURE_DIM = 6
HIDDEN_DIM = 128
SERVER_FEATURE_DIM = 20

encoder = DAGEncoder(
    task_feature_dim=TASK_FEATURE_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=2,
    encoder_type='gcn',
    server_feature_dim=SERVER_FEATURE_DIM
)
encoder.eval()

# Three graphs with DIFFERENT node counts (8, 12, 20)
graphs = []
for n in [8, 12, 20]:
    x = torch.rand(n, TASK_FEATURE_DIM)
    # Random directed edges (roughly a chain)
    src = torch.arange(0, n - 1)
    dst = torch.arange(1, n)
    edge_index = torch.stack([src, dst], dim=0)
    graphs.append(Data(x=x, edge_index=edge_index, num_nodes=n))

batch = Batch.from_data_list(graphs)

# Dummy task_features tensor
task_features_dummy = torch.zeros(3, 20, TASK_FEATURE_DIM)
server_features = torch.rand(3, SERVER_FEATURE_DIM)

with torch.no_grad():
    encoded_tasks, context = encoder(
        task_features=task_features_dummy,
        graph_batch=batch,
        server_features=server_features
    )

assert context.shape == (3, HIDDEN_DIM * 2), f"Bad context shape: {context.shape}"
print(f"PASS — context shape: {context.shape}  (expected [3, {HIDDEN_DIM*2}])")
print(f"PASS — encoded_tasks shape: {encoded_tasks.shape}")


PASS — context shape: torch.Size([3, 256])  (expected [3, 256])
PASS — encoded_tasks shape: torch.Size([3, 20, 256])


## 2. Generate the Golden Test Dataset

> ⚠️ Run **once only**.  Never re-run between algorithm comparisons.

In [7]:
!python utils/generate_test_dataset.py --num_dags 20 --output data/test_dags.json

import json
with open("data/test_dags.json") as f:
    ds = json.load(f)
print(f"Dataset contains {len(ds)} entries")
print("First entry keys:", list(ds[0].keys()))

Generating Golden Dataset of 20 DAGs from meta_offloading_n...
Loaded 4 DAG graphs from data/meta_offloading_n/offload_random10
Loaded 4 graphs from data/meta_offloading_n/offload_random10
Loaded 4 DAG graphs from data/meta_offloading_n/offload_random20
Loaded 4 graphs from data/meta_offloading_n/offload_random20
Loaded 4 DAG graphs from data/meta_offloading_n/offload_random30
Loaded 4 graphs from data/meta_offloading_n/offload_random30
Loaded 4 DAG graphs from data/meta_offloading_n/offload_random40
Loaded 4 graphs from data/meta_offloading_n/offload_random40
Loaded 4 DAG graphs from data/meta_offloading_n/offload_random50
Loaded 4 graphs from data/meta_offloading_n/offload_random50

Dataset of 20 DAGs saved → data/test_dags.json
Dataset contains 20 entries
First entry keys: ['num_tasks', 'tasks', 'edges', 'adj_matrix']


## 2.5 Verification — Physics and Reward Overhaul

Confirms that the environment correctly implements dynamic server loads (Makespan) and diverse, context-sensitive rewards.

In [8]:
import sys, yaml, numpy as np
sys.path.insert(0, '.')
from env.base_offloading_env import TaskOffloadingEnv
from utils.dag_parser import DAGParser

with open('configs/default_config.yaml') as f:
    fc = yaml.safe_load(f)
cfg = {}
for sec in ('system','computing','energy','network','tasks'):
    cfg.update(fc.get('environment', fc).get(sec, {}))

env = TaskOffloadingEnv(cfg)
dags = DAGParser('data/meta_offloading_n/offload_random20').load_dataset(num_graphs=1)
env.reset(task_graph=dags[0])

print("--- Cell A: Server Load Dynamic Check ---")
snapshots = [env.server_available.copy()]
for i in range(dags[0]['num_tasks']):
    action = i % env.action_space.n
    obs, reward, done, info = env.step(action)
    snapshots.append(env.server_available.copy())
    print(f'Node {i:2d} → server={action} | available={env.server_available.round(3)} | reward={reward:.3f}')
    if done:
        break

all_same = all(np.allclose(snapshots[0], s) for s in snapshots)
print('\n❌ FAIL — server state static.' if all_same else '\n✅ PASS — server state changes dynamically.')
print(f'Makespan: {env.total_delay:.4f}s  Energy: {env.total_energy:.6f}J')

print("\n--- Cell B: Reward Diversity Check ---")
dags_sample = DAGParser('data/meta_offloading_n/offload_random20').load_dataset(num_graphs=5)
rewards_by_action = {a: [] for a in range(env.action_space.n)}
for dag in dags_sample:
    env.reset(task_graph=dag)
    for _ in range(dag['num_tasks']):
        for a in range(env.action_space.n):
            d, e = env._execute_offloading(a)
            rewards_by_action[a].append(env._calculate_reward(d, e))
            # Undo state mutation
            env.node_finish_times[env.topo_order[env.current_node_idx]] = 0
            env.node_assignments[env.topo_order[env.current_node_idx]] = -1
for a, rs in rewards_by_action.items():
    print(f'Action {a}: mean={np.mean(rs):.3f}  min={np.min(rs):.3f}  max={np.max(rs):.3f}')


Loaded 1 DAG graphs from data/meta_offloading_n/offload_random20
--- Cell A: Server Load Dynamic Check ---
Node  0 → server=0 | available=[0.022 0.    0.    0.    0.   ] | reward=-0.004
Node  1 → server=1 | available=[0.022 0.004 0.    0.    0.   ] | reward=-5.000
Node  2 → server=2 | available=[0.022 0.004 0.005 0.    0.   ] | reward=-5.000
Node  3 → server=3 | available=[0.022 0.004 0.005 0.001 0.   ] | reward=-5.000
Node  4 → server=4 | available=[0.022 0.004 0.005 0.001 0.002] | reward=-5.000
Node  5 → server=0 | available=[0.066 0.004 0.005 0.001 0.002] | reward=-0.009
Node  6 → server=1 | available=[0.066 0.008 0.005 0.001 0.002] | reward=-5.000
Node  7 → server=2 | available=[0.066 0.008 0.012 0.001 0.002] | reward=-5.000
Node  8 → server=3 | available=[0.066 0.008 0.012 0.004 0.002] | reward=-5.000
Node  9 → server=4 | available=[0.066 0.008 0.012 0.004 0.01 ] | reward=-5.000
Node 10 → server=0 | available=[0.071 0.008 0.012 0.004 0.01 ] | reward=-0.001
Node 11 → server=1 | ava

## 3. Quick Training Smoke Test — TAMPO-GCN and TAMPO-LSTM (1 iteration)

Checks that the full training loop executes without shape errors for both algorithms.

In [9]:
import yaml, sys
sys.path.insert(0, '.')

with open('configs/default_config.yaml') as f:
    full_config = yaml.safe_load(f)

from env.base_offloading_env import TaskOffloadingEnv
from algorithms.rl.tampo import TAMPOFramework

cfg = {}
env_cfg = full_config.get('environment', full_config)
for sec in ('system','computing','energy','network','tasks'):
    cfg.update(env_cfg.get(sec, {}))
cfg['reward'] = env_cfg.get('reward', {})

env = TaskOffloadingEnv(cfg)

# Load training graphs from raw data folders (NEVER train on test_dags.json!)
from utils.dag_parser import DAGParser
dag_parser = DAGParser(data_folder='data/meta_offloading_20/offload_random20_1')
task_graphs = dag_parser.load_dataset(num_graphs=50)

tasks_for_env = []
for dag in task_graphs:
    tasks_for_env.append({
        'num_tasks': dag['num_tasks'],
        'tasks': dag['tasks'],
        'edges': dag['edges'],
        'adj_matrix': dag['adj_matrix'],
        'size': sum((t['data_size'] for t in dag['tasks']), start=0),
        'cycles': sum((t['cycles'] for t in dag['tasks']), start=0)
    })
env.load_task_dataset(tasks_for_env)

for enc_type in ['gcn', 'lstm']:
    print(f'\n======================================')
    print(f'Testing TAMPO-{enc_type.upper()}')
    print(f'======================================')
    tampo_cfg = full_config['algorithms']['tampo'].copy()
    tampo_cfg['encoder_type'] = enc_type
    tampo_cfg['num_meta_iterations'] = 1  # just 1 iteration for smoke test

    framework = TAMPOFramework(env, tampo_cfg)
    results = framework.train(num_iterations=1, meta_batch_size=2, checkpoint_path=f'models/tampo_{enc_type}_checkpoint.pth')
    
    import os
    os.makedirs('models', exist_ok=True)
    framework.save(f'models/tampo_{enc_type}_checkpoint.pth')
    print(f'SMOKE TEST PASSED for TAMPO-{enc_type.upper()} — checkpoint saved.')


Loaded 50 DAG graphs from data/meta_offloading_20/offload_random20_1
Loaded 50 tasks into environment dataset

Testing TAMPO-GCN
🔧 Using device: cuda
✓ Initialized new TAM-PO model with neutral decoder initialization

🚀 TAM-PO Meta-Training
  Iterations: 1
  Meta-batch size: 2
  Starting from iteration: 0
  Inner LR: 0.01
  Meta LR: 0.0003

  Iter   1/1 | Loss: -0.0522 | Avg(10): -0.0522 | Best: -0.0522

✓ Training complete!
  Total iterations: 1
  Final loss: -0.0522
  Best loss: -0.0522
  Model saved to: models/tampo_gcn_checkpoint.pth
✓ Model saved to models/tampo_gcn_checkpoint.pth
SMOKE TEST PASSED for TAMPO-GCN — checkpoint saved.

Testing TAMPO-LSTM
🔧 Using device: cuda
✓ Initialized new TAM-PO model with neutral decoder initialization

🚀 TAM-PO Meta-Training
  Iterations: 1
  Meta-batch size: 2
  Starting from iteration: 0
  Inner LR: 0.01
  Meta LR: 0.0003

  CuDNN disabled for TAMPO LSTM-style meta-training to support second-order gradients.
  Iter   1/1 | Loss: 0.0050 | Avg(

## 3.5 Complete Training — TAMPO-GCN and TAMPO-LSTM

In [11]:
# ── 5. Full Training Run (Long) ────────────────────────────────
# Note: This will take a while! Change num_iterations to 100 or 200 for proper convergence.

for enc_type in ['gcn', 'lstm']:
    print(f'\n======================================')
    print(f'Full Training TAMPO-{enc_type.upper()}')
    print(f'======================================')
    
    tampo_cfg = {**full_config['training'], **full_config['algorithms']['tampo']}
    tampo_cfg['encoder_type'] = enc_type
    
    framework = TAMPOFramework(env, tampo_cfg)
    
    # Using more iterations and larger meta_batch_size for real training
    results = framework.train(
        num_iterations=200, 
        meta_batch_size=10, 
        checkpoint_path=f'models/tampo_{enc_type}_checkpoint.pth'
    )
    
    framework.save(f'models/tampo_{enc_type}_checkpoint.pth')
    print(f'FULL TRAINING COMPLETE for TAMPO-{enc_type.upper()}')


KeyboardInterrupt: 

## 4. Run Benchmark on Trained Models

Compares both trained algorithms against the same test dataset.

In [ ]:
import os, sys
sys.path.insert(0, '.')

# Run benchmark for both algorithms
!python benchmark.py \
    --algorithms TAMPO_GCN TAMPO_LSTM \
    --checkpoint_dir models/ \
    --dataset_path data/test_dags.json \
    --output_dir results/


## 5. Download Results

In [ ]:
import os
import glob
from google.colab import files

# Find the most recently created run directory
run_dirs = sorted(glob.glob('results/run_*'), reverse=True)
if run_dirs:
    latest_run = run_dirs[0]
    for fname in ['comparison_bar.png', 'pareto_front.png', 'benchmark_results.csv']:
        path = os.path.join(latest_run, fname)
        if os.path.exists(path):
            files.download(path)
            print(f'Downloaded {path}')
        else:
            print(f'Not found: {path}')
else:
    print('No results found. Run benchmark first.')
